<a href="https://colab.research.google.com/github/ksenia980/practice/blob/main/3%20%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pandas -q

import requests
import pandas as pd

NGROK_URL = "https://numinous-overly-trent.ngrok-free.dev"
API_URL = f"{NGROK_URL}/api/v1"

print("Проверка подключения к API")
response = requests.get(f"{API_URL}/types/")
print(f"Статус: {response.status_code}")

data = response.json()
print(f"Тип данных: {type(data)}")

if isinstance(data, dict):
    print(f"Ключи словаря: {list(data.keys())}")

    if 'items' in data:
        types_data = data['items']
    elif 'data' in data:
        types_data = data['data']
    else:

        types_data = list(data.values())
else:
    types_data = data

print(f"Загружено {len(types_data)} типов экспонатов")
if types_data:
    print(f"Пример: {types_data[0]}")

Проверка подключения к API
Статус: 200
Тип данных: <class 'dict'>
Ключи словаря: ['data', 'sql']
Загружено 15 типов экспонатов
Пример: {'name': 'Картина', 'id': 1}


In [ ]:
import requests
NGROK_URL = "https://numinous-overly-trent.ngrok-free.dev"
API_URL = f"{NGROK_URL}/api/v1"

# Функция для извлечения данных из API
def get_data(response):
    data = response.json()
    if isinstance(data, dict) and 'data' in data:
        return data['data']
    return data

response = requests.get(f"{API_URL}/types/")
types_data = get_data(response)
print(f"API работает! Загружено {len(types_data)} типов\n")

# ЗАДАНИЕ 1: Типы со словом "изделие"
print("ЗАДАНИЕ 1: Типы экспонатов, содержащие слово 'изделие'")

response = requests.get(f"{API_URL}/types/")
types = get_data(response)

found = []
for t in types:
    if "изделие" in t['name'].lower():
        found.append(t['name'])
        print(f" {t['name']}")

if not found:
    print("Типов со словом 'изделие' не найдено")
    print("\nВсе доступные типы:")
    for t in types:
        print(f"  - {t['name']}")
else:
    print(f"\n Найдено: {len(found)} типов\n")

# ЗАДАНИЕ 2: Типы и количество экспонатов

print("ЗАДАНИЕ 2: Типы экспонатов и количество предметов")

wings = get_data(requests.get(f"{API_URL}/wings/"))
types = get_data(requests.get(f"{API_URL}/types/"))

print(f"Экспонатов: {len(wings)} | Типов: {len(types)}")
type_names = {t['id']: t['name'] for t in types}

# Подсчет
count_dict = {}
for wing in wings:
    type_id = wing.get('type_id')
    if type_id:
        count_dict[type_id] = count_dict.get(type_id, 0) + 1

print(f"\n{'Тип экспоната':<35} | Количество")
print("-" * 50)
for type_id, count in sorted(count_dict.items(), key=lambda x: x[1], reverse=True):
    name = type_names.get(type_id, f"Тип {type_id}")
    print(f"{name:<35} | {count}")

print(f"\n Всего типов с экспонатами: {len(count_dict)}\n")


# ЗАДАНИЕ 3: Топ-10 самых дорогих (>30000)
print("ЗАДАНИЕ 3: Топ-10 самых дорогих экспонатов (>30000 руб.)")
moves = get_data(requests.get(f"{API_URL}/moves/"))
wings = get_data(requests.get(f"{API_URL}/wings/"))
places = get_data(requests.get(f"{API_URL}/places/"))

print(f"Перемещений: {len(moves)} | Экспонатов: {len(wings)} | Мест: {len(places)}")
wings_dict = {w['id']: w for w in wings}
places_dict = {p['id']: p for p in places}
# Фильтр >30000
expensive = [m for m in moves if m.get('price', 0) > 30000]
expensive.sort(key=lambda x: x['price'], reverse=True)
top_10 = expensive[:10]

print(f"\n Перемещений дороже 30000 руб.: {len(expensive)}")
if not top_10:
    print("\n Нет перемещений дороже 30000 руб.")
    print("\nСамые дорогие из доступных:")
    all_sorted = sorted(moves, key=lambda x: x['price'], reverse=True)
    for i, move in enumerate(all_sorted[:5], 1):
        wing = wings_dict.get(move['wing_id'], {})
        print(f"  {i}. {wing.get('name', 'Без названия')} — {move['price']:,.2f} руб.")
else:
    print(f"\n ТОП-{len(top_10)} самых дорогих:\n")
    for i, move in enumerate(top_10, 1):
        wing = wings_dict.get(move['wing_id'], {})
        place = places_dict.get(move['place_id'], {})

        wing_name = wing.get('name', 'Без названия')
        price = move['price']
        location = place.get('location', 'Неизвестно')
        date = move.get('dt', 'Нет даты')
        if len(date) > 10:
            date = date[:10]

        print(f"{i:2}. {wing_name}")
        print(f"     Стоимость: {price:>12,.2f} руб.")
        print(f"     Место: {location}")
        print(f"     Дата: {date}")
        print()



API работает! Загружено 15 типов

ЗАДАНИЕ 1: Типы экспонатов, содержащие слово 'изделие'
 Ювелирное изделие

 Найдено: 1 типов

ЗАДАНИЕ 2: Типы экспонатов и количество предметов
Экспонатов: 100 | Типов: 15

Тип экспоната                       | Количество
--------------------------------------------------
Монета                              | 12
Ювелирное изделие                   | 10
Картина                             | 9
Книга                               | 9
Фотография                          | 9
Текстиль                            | 9
Мебель                              | 7
Рукопись                            | 6
Скульптура                          | 5
Прибор                              | 5
Оружие                              | 5
Керамика                            | 5
Инструмент                          | 4
Источник вдохновения                | 3
Артефакт                            | 2

 Всего типов с экспонатами: 15

ЗАДАНИЕ 3: Топ-10 самых дорогих экспонатов (>30000 руб.)
П